# Model Training - ResNet50

Este notebook entrena un modelo **ResNet50** con transfer learning para la clasificación de defectos en acero.

**Objetivo:** Segundo modelo del ensemble (≥90% individual)

**Arquitectura:**
- Base: ResNet50 pre-entrenado en ImageNet
- Input: 224x224x3 (RGB)
- Data Augmentation
- Class Weights
- Two-phase training

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Configuration
BASE_DIR = Path(r'C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos')
DATA_DIR = BASE_DIR / 'data' / 'processed'
MODEL_DIR = BASE_DIR / 'ml_models' / 'resnet50'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
NUM_CLASSES = 4

print(f"Data directory: {DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")

In [ ]:
# Data generators
train_datagen = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.7, 1.3],
    zoom_range=0.15,
    shear_range=0.1,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator()

train_generator = train_datagen.flow_from_directory(
    DATA_DIR / 'train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_generator = val_test_datagen.flow_from_directory(
    DATA_DIR / 'val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    DATA_DIR / 'test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"Train: {train_generator.samples}, Val: {val_generator.samples}, Test: {test_generator.samples}")

In [ ]:
# Class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))

print("Class weights:")
for idx, weight in class_weight_dict.items():
    class_name = list(train_generator.class_indices.keys())[list(train_generator.class_indices.values()).index(idx)]
    print(f"  {class_name}: {weight:.3f}")

In [ ]:
def build_resnet50_model(input_shape=(224, 224, 3), num_classes=4):
    """
    Build ResNet50 model with transfer learning
    """
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    base_model.trainable = False
    
    inputs = keras.Input(shape=input_shape)
    x = keras.applications.resnet.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    return model, base_model

model, base_model = build_resnet50_model(
    input_shape=(*IMG_SIZE, 3),
    num_classes=NUM_CLASSES
)

print(f"Total parameters: {model.count_params():,}")
print(f"Trainable parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
)

print("Model compiled!")

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    ModelCheckpoint(filepath=str(MODEL_DIR / 'resnet50_best.keras'), monitor='val_accuracy', save_best_only=True, verbose=1),
    TensorBoard(log_dir=str(MODEL_DIR / 'logs' / datetime.now().strftime('%Y%m%d-%H%M%S')))
]

In [ ]:
print("="*80)
print("PHASE 1: Training with frozen base model")
print("="*80)

history_phase1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Phase 1 completed!")

In [ ]:
print("="*80)
print("PHASE 2: Fine-tuning")
print("="*80)

base_model.trainable = True

# Unfreeze last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE/10),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
)

history_phase2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Phase 2 completed!")

In [ ]:
print("="*80)
print("EVALUATING ON TEST SET")
print("="*80)

best_model = keras.models.load_model(MODEL_DIR / 'resnet50_best.keras')
test_loss, test_acc, test_auc, test_precision, test_recall = best_model.evaluate(test_generator, verbose=1)

print(f"\n📊 TEST RESULTS:")
print(f"  Accuracy:  {test_acc*100:.2f}%")
print(f"  AUC:       {test_auc:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall:    {test_recall:.4f}")
print(f"  F1-Score:  {2 * (test_precision * test_recall) / (test_precision + test_recall):.4f}")

In [ ]:
test_generator.reset()
y_pred_probs = best_model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

class_names = list(test_generator.class_indices.keys())
print("\nCLASSIFICATION REPORT:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', 
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - ResNet50', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
combined_history = {
    'accuracy': history_phase1.history['accuracy'] + history_phase2.history['accuracy'],
    'val_accuracy': history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy'],
    'loss': history_phase1.history['loss'] + history_phase2.history['loss'],
    'val_loss': history_phase1.history['val_loss'] + history_phase2.history['val_loss']
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(combined_history['accuracy'], label='Train', linewidth=2)
axes[0].plot(combined_history['val_accuracy'], label='Val', linewidth=2)
axes[0].axvline(x=len(history_phase1.history['accuracy']), color='red', linestyle='--')
axes[0].set_title('Model Accuracy - ResNet50', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(combined_history['loss'], label='Train', linewidth=2)
axes[1].plot(combined_history['val_loss'], label='Val', linewidth=2)
axes[1].axvline(x=len(history_phase1.history['loss']), color='red', linestyle='--')
axes[1].set_title('Model Loss - ResNet50', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'training_history.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
best_model.save(MODEL_DIR / 'resnet50_final.keras')

metrics = {
    'model_name': 'ResNet50',
    'test_accuracy': float(test_acc),
    'test_auc': float(test_auc),
    'test_precision': float(test_precision),
    'test_recall': float(test_recall),
    'test_f1': float(2 * (test_precision * test_recall) / (test_precision + test_recall)),
    'training_date': datetime.now().isoformat()
}

with open(MODEL_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("\n✅ ResNet50 training complete!")
print(f"Test Accuracy: {test_acc*100:.2f}%")